In [ ]:
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import scikit_posthocs as sp

In [ ]:
df_rfm = pd.read_csv('RFM_base.csv')
df_monthly = pd.read_csv('agg_m.csv')
df_i = pd.read_csv('agg_i.csv')

In [ ]:
# RFM

# R - Recency (давность покупки), F - Frequency (частота покупок), M - Monetary (сумма покупок)

labels_r = [5, 4, 3, 2, 1]
labels_m = [1, 2, 3, 4, 5]

df_rfm['R_score'] = pd.qcut(
    df_rfm['R'],
    q=5,
    labels=labels_r
)

df_rfm['F_score'] = pd.qcut(
    df_rfm['F'],
    q=5,
    labels=False,
    duplicates='drop'
) + 1

df_rfm['M_score'] = pd.qcut(
    df_rfm['M'],
    q=5,
    labels=labels_m
)

df_rfm['RFM_score'] = (
    df_rfm['R_score'].astype(str) +
    df_rfm['F_score'].astype(str) +
    df_rfm['M_score'].astype(str)
)

def segment(x):
    r, f, m = x['R_score'], x['F_score'], x['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Лояльные'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'В зоне оттока'
    elif r >= 4 and f <= 2:
        return 'Новые'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Потерянные'
    else:
        return 'Прочие'

df_rfm['Segment'] = df_rfm.apply(segment, axis=1)

df_rfm.head()

In [ ]:
segments = df_rfm.groupby('Segment').agg(
    Count=('Customer ID', 'count'),
    AVG_R=('R', 'mean'),
    AVG_F=('F', 'mean'),
    AVG_M=('M', 'mean'),
    Total_M=('M', 'sum')
).round(1).reset_index()

segments['Revenue_Share_%'] = (
    segments['Total_M']/df_rfm['M'].sum()*100
).round(1)

segments['Count_Share_%'] = (
    segments['Count']/segments['Count'].sum()*100
).round(1)
segments = segments.sort_values('Count_Share_%', ascending=False)
segments.head(50)

# Прочие = 43,9% - самый крупный сегмент, по сути "мусорная корзина" для тех, кто не попал под чёткие правила.
# Соответственно, лучше ввести подсегменты внутри "Прочих"

In [ ]:
def segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Лояльные'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'В зоне оттока'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'Уходящие'
    elif r >= 4 and f <= 2 and m >= 3:
        return 'Новые с высоким чеком'
    elif r >= 4 and f <= 2:
        return 'Новые'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Потерянные'
    elif f >= 4 and m <= 2:
        return 'Частые с низким чеком'
    elif r == 3 and f == 3 and m == 3:
        return 'Стабильные средние'
    else:
        return 'Прочие'

df_rfm['Segment'] = df_rfm.apply(segment, axis=1)

segments = df_rfm.groupby('Segment').agg(
    Count=('Customer ID', 'count'),
    AVG_R=('R', 'mean'),
    AVG_F=('F', 'mean'),
    AVG_M=('M', 'mean'),
    Total_M=('M', 'sum')
).round(1).reset_index()

segments['Revenue_Share_%'] = (
    segments['Total_M']/df_rfm['M'].sum()*100
).round(1)

segments['Count_Share_%'] = (
    segments['Count']/segments['Count'].sum()*100
).round(1)

segments = segments.sort_values('Count_Share_%', ascending=False)

segments.head(40)

In [ ]:
# Статистика

group_df_i = df_i.groupby('Country').agg(
    Count=('Invoice', 'count')
).reset_index()
group_df_i['Rank'] = group_df_i['Count'].rank(method='first', ascending=False)

group_df_i = group_df_i.sort_values('Rank')
group_df_i = (
    group_df_i[group_df_i['Rank'] <= 6]
)

group_df_i.head(50)

In [ ]:
countries = ['United Kingdom', 'Germany', 'EIRE', 'France', 'Netherlands', 'Spain']

df_i_filtered = df_i[df_i['Country'].isin(countries)].copy()

# df_i_filtered = df_i[df_i['Country'].isin(countries)].drop('Invoice', axis=1)

# Смотрим на отрицательную выручку на уровне счёта
negative_revenue = df_i_filtered[df_i_filtered['Revenue'] < 0]
print('Количество счетов с отрицательной выручкой:', len(negative_revenue))

# Исключаем счета с отрицательной выручкой — это не реальные продажи,
# а служебные корректировки с отрицательной ценой при положительном количестве
df_i_filtered_clean = df_i_filtered[df_i_filtered['Revenue'] > 0].copy()

print(f'Исключено счетов: {len(df_i_filtered) - len(df_i_filtered_clean)}')

uk = df_i_filtered_clean[df_i_filtered_clean['Country'] == 'United Kingdom']['Revenue']
germany = df_i_filtered_clean[df_i_filtered_clean['Country'] == 'Germany']['Revenue']
france = df_i_filtered_clean[df_i_filtered_clean['Country'] == 'France']['Revenue']
eire = df_i_filtered_clean[df_i_filtered_clean['Country'] == 'EIRE']['Revenue']
spain = df_i_filtered_clean[df_i_filtered_clean['Country'] == 'Spain']['Revenue']
netherlands = df_i_filtered_clean[df_i_filtered_clean['Country'] == 'Netherlands']['Revenue']

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, country in enumerate(countries):
    data = df_i_filtered_clean[df_i_filtered_clean['Country'] == country]['Revenue']
    axes[i].hist(data, bins=50)
    axes[i].set_title(country)
    axes[i].set_xlabel('Revenue')

plt.tight_layout()
plt.show()

In [ ]:
df_i_filtered_clean.sort_values('Revenue', ascending=False).head()

In [ ]:
df_i_filtered_clean.loc[41908]

In [ ]:
# Строка с аномальным количеством (80 995 шт. одного товара
# PAPER CRAFT, LITTLE BIRDIE в одном счёте)
# похожа на ошибку ввода данных, а не на легитимный опт —
# исключаем этот счёт из анализа по странам
anomaly_invoice = '581483'

df_i_filtered_clean = df_i_filtered_clean[df_i_filtered_clean['Invoice'] != anomaly_invoice]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, country in enumerate(countries):
    data = df_i_filtered_clean[df_i_filtered_clean['Country'] == country]['Revenue']
    axes[i].hist(data, bins=50)
    axes[i].set_title(country)
    axes[i].set_xlabel('Revenue')

plt.tight_layout()
plt.show()

In [ ]:
result_levene = stats.levene(uk, germany, france, eire, spain, netherlands)

print('statistic:', result_levene.statistic)
print('pvalue:', result_levene.pvalue)

In [ ]:
# H0: медиана одинакова во всех сравниваемых странах.
# H1: хотя бы в одной стране медиана статистически значимо отличается.

result_kruskal = stats.kruskal(uk, germany, france, eire, spain, netherlands)

print('Kruskal-Wallis statistic:', result_kruskal.statistic)
print('Kruskal-Wallis pvalue:', result_kruskal.pvalue)

In [ ]:
# тест Данна

result_dunn = sp.posthoc_dunn(
    df_i_filtered_clean,
    val_col='Revenue',
    group_col='Country',
    p_adjust='bonferroni'
)

result_dunn

In [ ]:
country_stats = (
    df_i_filtered_clean
    .groupby('Country')['Revenue']
    .agg(
        Median_Revenue='median',
        Mean_Revenue='mean',
        Count='count'
    ).reset_index()
)

country_stats.head(10)

In [ ]:
# ТРЕНД

# Убираем 2011 12, т.к. данные всего за 9 дней в этом месяце.

df_monthly = df_monthly.query('Date < "2011-12"')

df_y_m = df_monthly.copy()

df_y_m = df_monthly.assign(
    Year=df_monthly['Date'].str[0:4].astype(int),
    Month=df_monthly['Date'].str[5:7].astype(int)
)

df_y_m = df_y_m.drop(columns=['Date'])

df_y_m.head(50)

In [ ]:
comparison = df_y_m[df_y_m['Year'].isin([2010, 2011])].pivot_table(
    index='Month',
    columns='Year',
    values='Revenue'
)

comparison['Growth_%'] = (
    (comparison[2011] - comparison[2010])
    / comparison[2010] * 100
).round(2)

comparison.head(50)

In [ ]:
df_year_2010 = df_y_m[df_y_m['Year'] == 2010].copy()
df_year_2011 = df_y_m[df_y_m['Year'] == 2011].copy()

plt.figure(figsize=(15, 6))
plt.plot(df_year_2010['Month'], df_year_2010['Revenue'], marker='o', label='2010 г.')
plt.plot(df_year_2011['Month'], df_year_2011['Revenue'], marker='o', label='2011 г.', color='green')
plt.xticks(range(1, 13))
plt.legend()
plt.title('Сравнение сезонности по месяцам: 2010 vs 2011')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.grid()
plt.show()

In [ ]:
# df_y_m.to_csv('trend.csv', index=False)

In [ ]:
# segments.to_csv('segment.csv', index=False)

In [ ]:
# country_stats.to_csv('country_stats.csv', index=False)

In [ ]:
# df_i_filtered_clean.to_csv('df_i_filtered_clean.csv', index=False)

In [ ]:
# df_rfm.to_csv('rfm.csv', index=False)